# Topic: Token Generator System - CSPRNG secrets generation, HMAC signed payloads, and session verification checks
 
## 1. CSPRNG VS PRNG
- **Pseudo-Random Number Generators (PRNGs)**: Python's standard `random` module uses the Mersenne Twister algorithm. It is highly predictable once the state is observed, making it a critical security risk for generating tokens or keys.
- **Cryptographically Secure PRNGs (CSPRNGs)**: Python's **`secrets`** module interacts directly with OS entropy sources (`/dev/urandom` or Windows CryptGenRandom). It is mathematically unpredictable, making it suitable for security operations.
 
## 2. SIGNED TOKENS (HMAC-SHA256)
- To authenticate sessions without database lookups on every request, we generate a signed token.
- The token contains:
  1. **Payload**: JSON/Text representing user state and an **Expiration Timestamp**.
  2. **HMAC Signature**: A Hash-based Message Authentication Code computed using a secret server key:
     $$\text{Signature} = \text{HMAC}(\text{SecretKey}, \text{Payload})$$
- **Tamper Proof**: If an attacker attempts to modify the payload (e.g., change user ID or extend expiration), the server's verification check will fail because the computed signature will not match the token's signature.
 
---


In [ ]:
import secrets
import hmac
import hashlib
import time
import json
import base64

class SignedTokenSystem:
    def __init__(self, server_secret_key):
        self.secret_key = server_secret_key.encode('utf-8')

    def generate_token(self, user_id, lifespan_seconds=10):
        """Generates an HMAC-signed base64 token containing user_id and expiration."""
        # Calculate expiration timestamp
        expires_at = time.time() + lifespan_seconds
        
        # Build payload
        payload_dict = {
            "user_id": user_id,
            "exp": expires_at,
            "nonce": secrets.token_hex(8)  # Anti-replay attack nonce
        }
        
        # Base64 encode the payload string
        payload_bytes = json.dumps(payload_dict).encode('utf-8')
        payload_b64 = base64.urlsafe_b64encode(payload_bytes).decode('utf-8').rstrip('=')
        
        # Calculate HMAC signature
        signature = hmac.new(
            self.secret_key,
            payload_b64.encode('utf-8'),
            hashlib.sha256
        ).digest()
        
        signature_b64 = base64.urlsafe_b64encode(signature).decode('utf-8').rstrip('=')
        
        # Complete token format: [Payload_B64].[Signature_B64]
        return f"{payload_b64}.{signature_b64}"

    def verify_token(self, token_str):
        """Verifies HMAC signature and expiration. Returns payload dict if valid."""
        parts = token_str.split('.')
        if len(parts) != 2:
            print("[!] Token Format Error")
            return None
            
        payload_b64, signature_b64 = parts[0], parts[1]
        
        # Recompute signature
        expected_sig = hmac.new(
            self.secret_key,
            payload_b64.encode('utf-8'),
            hashlib.sha256
        ).digest()
        
        expected_sig_b64 = base64.urlsafe_b64encode(expected_sig).decode('utf-8').rstrip('=')
        
        # Constant-time comparison (prevents timing attacks!)
        if not hmac.compare_digest(signature_b64, expected_sig_b64):
            print("[!] Verification Failed: Signature mismatch!")
            return None
            
        # Decode and inspect payload
        try:
            # Re-add padding bytes for base64 decoding
            padding = '=' * (4 - (len(payload_b64) % 4))
            payload_bytes = base64.urlsafe_b64decode(payload_b64 + padding)
            payload = json.loads(payload_bytes.decode('utf-8'))
        except Exception:
            print("[!] Decoding Error")
            return None
            
        # Check Expiration
        if time.time() > payload["exp"]:
            print("[!] Verification Failed: Token has expired!")
            return None
            
        return payload



In [ ]:
# Testing the Token System
print("--- Initializing Token Engine ---")
# Instantiate server signed engine
server_key = "ServerSecretPassphraseXYZ"
token_sys = SignedTokenSystem(server_key)

# 1. Generate Token for User 'alice_dev' valid for 2 seconds
token = token_sys.generate_token("alice_dev", lifespan_seconds=2)
print(f"Generated Token: {token}")

# 2. Verify immediately (should be valid)
print("\n--- Immediate Verification ---")
result = token_sys.verify_token(token)
print(f"Verification Output: {result}")



In [ ]:
# 3. Simulating Tampering
print("\n--- Simulating Tampering Attack ---")
# Split token
payload_b64, sig_b64 = token.split('.')
# Decode payload and edit user_id to "admin"
padding = '=' * (4 - (len(payload_b64) % 4))
payload_bytes = base64.urlsafe_b64decode(payload_b64 + padding)
payload_dict = json.loads(payload_bytes.decode('utf-8'))

payload_dict["user_id"] = "admin"  # Attempt privilege escalation!

# Re-encode tampered payload
tampered_bytes = json.dumps(payload_dict).encode('utf-8')
tampered_b64 = base64.urlsafe_b64encode(tampered_bytes).decode('utf-8').rstrip('=')

# Re-assemble token using original signature
tampered_token = f"{tampered_b64}.{sig_b64}"
print(f"Tampered Token: {tampered_token}")

# Verify tampered token
token_sys.verify_token(tampered_token)
# Expected: Signature verification fails!



In [ ]:
# 4. Simulating Expiration
print("\n--- Simulating Expiration ---")
time.sleep(3.0)  # Wait for token to expire (lifespan was 2 seconds)
token_sys.verify_token(token)
# Expected: Token has expired!
